# STRIVE: external-data model comparison in hosted Google Colab

Use **Runtime → Change runtime type → hosted CPU or GPU**, then **Runtime → Run all**. Upload `data/training/reviewed-answers.jsonl` when prompted. The notebook downloads a pinned public business/economics QNLI subset, obtains hosted Gemini embeddings, compares two models on the same held-out business families, and downloads the results. No model runs on your computer. Deployment is separate.

This is answer relevance, not business-success prediction. More data does not guarantee an improvement. QNLI negatives mean “does not contain an answer”; this is auxiliary supervision, not expert-labelled entrepreneur off-topic data. Attribution: [SetFit QNLI](https://huggingface.co/datasets/SetFit/qnli), derived via [GLUE](https://huggingface.co/datasets/nyu-mll/glue#qnli) from [SQuAD](https://rajpurkar.github.io/SQuAD-explorer/) (CC BY-SA 4.0).

In [ ]:
import importlib.util
if importlib.util.find_spec('google.colab') is None:
    raise RuntimeError('Use a hosted Google Colab runtime. Local training is disabled.')
%pip -q install numpy requests scikit-learn


In [ ]:
from pathlib import Path
SOURCES = {'train_cloud.py': '"""Run in Google Colab. No training is permitted in a local runtime."""\nimport importlib.util\nif importlib.util.find_spec(\'google.colab\') is None:\n    raise RuntimeError(\'Open cloud_training.ipynb in Google Colab and use a hosted runtime. Local training is disabled.\')\n\nimport collections, datetime, hashlib, json, os, time\nfrom pathlib import Path\nimport numpy as np\nimport requests\nfrom sklearn.linear_model import LogisticRegression\nfrom sklearn.metrics import classification_report, confusion_matrix, f1_score, recall_score\nfrom sklearn.model_selection import StratifiedGroupKFold\n\nFEATURE_VERSION = \'context-question-answer-v1\'\nEMBEDDING_MODEL = os.environ.get(\'GEMINI_EMBEDDING_MODEL\', \'gemini-embedding-001\')\nDIMENSION = 768\n\ndef features(row):\n    return f"Business: {row[\'businessContext\']}\\nQuestion: {row[\'question\']}\\nAnswer: {row[\'answer\']}"[:7000]\n\ndef embedding(text, key):\n    url = f\'https://generativelanguage.googleapis.com/v1beta/models/{EMBEDDING_MODEL}:embedContent\'\n    payload = {\'model\': f\'models/{EMBEDDING_MODEL}\', \'content\': {\'parts\': [{\'text\': text}]}, \'taskType\': \'CLASSIFICATION\', \'outputDimensionality\': DIMENSION}\n    for attempt in range(5):\n        response = requests.post(url, headers={\'x-goog-api-key\': key}, json=payload, timeout=60)\n        if response.status_code == 429 or response.status_code >= 500:\n            time.sleep(min(60, 5 * 2**attempt))\n            continue\n        response.raise_for_status()\n        vector = np.asarray(response.json()[\'embedding\'][\'values\'], dtype=float)\n        if len(vector) != DIMENSION or not np.isfinite(vector).all() or np.linalg.norm(vector) == 0:\n            raise ValueError(\'Invalid embedding returned; training stopped.\')\n        return vector / np.linalg.norm(vector)\n    raise RuntimeError(\'Embedding quota or provider availability prevented training. Retry later; no fake data was substituted.\')\n\ndef training_readiness(rows):\n    counts = collections.Counter(row[\'label\'] for row in rows)\n    groups = {str(row[\'groupId\']).strip() for row in rows if str(row[\'groupId\']).strip()}\n    checks = [\n        (\'Reviewed answers\', len(rows), 20),\n        (\'Business families\', len(groups), 4),\n        (\'Relevant answers (label 0)\', counts.get(0, 0), 5),\n        (\'Off-topic answers (label 1)\', counts.get(1, 0), 5),\n    ]\n    return {\'ready\': all(current >= required for _, current, required in checks),\n            \'checks\': [{\'name\': name, \'current\': current, \'required\': required,\n                        \'remaining\': max(0, required-current)} for name, current, required in checks]}\n\n\ndef check_training_file(path):\n    rows = []\n    for number, line in enumerate(Path(path).read_text(encoding=\'utf-8-sig\').splitlines(), 1):\n        if not line.strip():\n            continue\n        try:\n            row = json.loads(line)\n        except json.JSONDecodeError as error:\n            raise ValueError(f\'Line {number} is not valid JSON. Upload the reviewed JSONL export, not a PDF or archive.\') from error\n        rows.append(row)\n    required = [\'businessContext\', \'question\', \'answer\', \'label\', \'groupId\', \'language\']\n    for number, row in enumerate(rows, 1):\n        if not isinstance(row, dict) or not all(key in row for key in required) or type(row[\'label\']) is not int or row[\'label\'] not in [0, 1] or not isinstance(row[\'groupId\'], str) or not row[\'groupId\'].strip():\n            raise ValueError(\'Every reviewed row needs context, question, answer, label 0/1, language, and groupId.\')\n        if not all(isinstance(row[key], str) for key in [\'businessContext\', \'question\', \'answer\', \'language\']) or not row[\'question\'].strip() or not row[\'answer\'].strip():\n            raise ValueError(f\'Line {number} needs a non-empty question and answer, with text context and language.\')\n        row[\'groupId\'] = row[\'groupId\'].strip()\n        if row.get(\'example\') or str(row.get(\'submissionId\', \'\')).startswith(\'example-\'):\n            raise ValueError(\'Synthetic interface examples cannot be used as training evidence.\')\n    readiness = training_readiness(rows)\n    print(\'Training data checks (no model training or embedding requests yet):\')\n    for check in readiness[\'checks\']:\n        print(f"  {check[\'name\']}: {check[\'current\']} / {check[\'required\']} minimum")\n    if not readiness[\'ready\']:\n        missing = \'; \'.join(f"{check[\'remaining\']} more {check[\'name\'].lower()}" for check in readiness[\'checks\'] if check[\'remaining\'])\n        raise ValueError(f\'Training data is incomplete: {missing}. Add genuine reviewed answers, export a NEW reviewed-answers.jsonl, upload it again, and rerun this check. Keep related business variants together; do not duplicate rows or change correct labels just to pass.\')\n    seen = {}\n    for row in rows:\n        signature = hashlib.sha256(features(row).strip().lower().encode()).hexdigest()\n        if signature in seen:\n            if seen[signature] != row[\'groupId\']:\n                raise ValueError(\'An identical feature record crosses business groups. Fix group assignments before training.\')\n            raise ValueError(\'Duplicate question/answer features were found. Remove duplicates rather than counting repeated copies as reviewed examples.\')\n        seen[signature] = row[\'groupId\']\n    print(\'Minimum data checks passed. Group assignments and label correctness still need source review.\')\n    return rows\n\n\ndef train(path, api_key):\n    rows = check_training_file(path)\n    counts = collections.Counter(row[\'label\'] for row in rows)\n    groups = np.array([row[\'groupId\'] for row in rows])\n    data_hash = hashlib.sha256(json.dumps(rows, sort_keys=True, ensure_ascii=False).encode()).hexdigest()\n    cache = Path(\'embedding_cache.json\')\n    saved = json.loads(cache.read_text()) if cache.exists() else {}\n    vectors = []\n    for index, row in enumerate(rows):\n        key = hashlib.sha256((EMBEDDING_MODEL + FEATURE_VERSION + features(row)).encode()).hexdigest()\n        if key not in saved:\n            saved[key] = embedding(features(row), api_key).tolist()\n            cache.write_text(json.dumps(saved))\n        vectors.append(saved[key])\n        if (index+1) % 10 == 0:\n            print(f\'Embedded {index+1}/{len(rows)} reviewed answers\')\n    X, y = np.asarray(vectors), np.array([row[\'label\'] for row in rows])\n    splitter = StratifiedGroupKFold(n_splits=min(5, len(set(groups))), shuffle=True, random_state=42)\n    oof = np.full(len(rows), np.nan)\n    folds = []\n    for fold, (train_ids, test_ids) in enumerate(splitter.split(X, y, groups)):\n        if len(set(y[train_ids])) < 2:\n            folds.append({\'fold\': fold, \'skipped\': \'Training partition contains only one class.\'})\n            continue\n        assert not set(groups[train_ids]) & set(groups[test_ids])\n        model = LogisticRegression(C=1.0, class_weight=\'balanced\', max_iter=2000, random_state=42)\n        model.fit(X[train_ids], y[train_ids])\n        oof[test_ids] = model.predict_proba(X[test_ids])[:, 1]\n        folds.append({\'fold\': fold, \'trainRows\': len(train_ids), \'testRows\': len(test_ids), \'testGroups\': sorted(set(groups[test_ids]))})\n    valid = np.isfinite(oof)\n    if not valid.any():\n        raise ValueError(\'No valid grouped evaluation folds. Review the label distribution across groups.\')\n    predicted = (oof[valid] >= 0.5).astype(int)\n    metrics = {\'evaluation\': \'grouped out-of-fold; fixed C=1; not an independent external test\', \'rows\': len(rows), \'groups\': len(set(groups)), \'classCounts\': dict(counts), \'evaluatedRows\': int(valid.sum()), \'evaluationCoverage\': float(valid.mean()), \'macroF1\': float(f1_score(y[valid], predicted, average=\'macro\', zero_division=0)), \'offTopicRecall\': float(recall_score(y[valid], predicted, pos_label=1, zero_division=0)), \'confusionMatrix\': confusion_matrix(y[valid], predicted, labels=[0, 1]).tolist(), \'report\': classification_report(y[valid], predicted, labels=[0, 1], target_names=[\'relevant\', \'off_topic\'], output_dict=True, zero_division=0), \'advisoryThreshold\': 0.75, \'flagCoverage\': float(np.mean(oof[valid] >= 0.75)), \'abstentionRate\': float(np.mean((oof[valid] > 0.25) & (oof[valid] < 0.75))), \'folds\': folds, \'languages\': {}}\n    for language in sorted(set(row[\'language\'] for row in rows)):\n        mask = valid & np.array([row[\'language\'] == language for row in rows])\n        metrics[\'languages\'][language] = {\'rows\': int(mask.sum()), \'macroF1\': float(f1_score(y[mask], (oof[mask]>=0.5).astype(int), average=\'macro\', zero_division=0)) if mask.sum() >= 5 and len(set(y[mask])) == 2 else None}\n    final = LogisticRegression(C=1.0, class_weight=\'balanced\', max_iter=2000, random_state=42).fit(X, y)\n    artifact = {\'format\': \'strive-relevance-v1\', \'modelVersion\': f\'relevance-{data_hash[:12]}\', \'featureVersion\': FEATURE_VERSION, \'embeddingModel\': EMBEDDING_MODEL, \'dimension\': DIMENSION, \'coefficients\': final.coef_[0].tolist(), \'intercept\': float(final.intercept_[0]), \'classes\': [0, 1], \'trainingDataHash\': data_hash, \'trainedAt\': datetime.datetime.now(datetime.timezone.utc).isoformat(), \'metrics\': metrics, \'advisoryOnly\': True}\n    logits = X[:5] @ final.coef_[0] + final.intercept_[0]\n    probs = 1 / (1 + np.exp(-logits))\n    assert np.allclose(probs, final.predict_proba(X[:5])[:, 1])\n    Path(\'model.json\').write_text(json.dumps(artifact, indent=2), encoding=\'utf-8\')\n    Path(\'evaluation.json\').write_text(json.dumps(metrics, indent=2), encoding=\'utf-8\')\n    Path(\'parity-vectors.json\').write_text(json.dumps({\'vectors\': X[:5].tolist(), \'expectedOffTopicScores\': probs.tolist()}), encoding=\'utf-8\')\n    print(json.dumps({k: metrics[k] for k in [\'rows\',\'groups\',\'macroF1\',\'offTopicRecall\',\'evaluationCoverage\',\'abstentionRate\']}, indent=2))\n    print(\'Saved actual trained model, evaluation results, and inference parity vectors. Small-data results are experimental.\')\n    return artifact\n', 'external_data.py': '"""Download and prepare public auxiliary data. No model training or inference."""\nimport argparse\nimport collections\nimport hashlib\nimport json\nimport re\nimport urllib.request\nfrom pathlib import Path\n\nDATASET = \'SetFit/qnli\'\nREVISION = \'e08ed8239c319cf7f2d2ab393d1b90e276c39570\'\nBASE = f\'https://huggingface.co/datasets/{DATASET}/resolve/{REVISION}\'\nHASHES = {\n    \'train\': \'9dc5d6276c1f71b43817377d8eb9dc3e751dd9e881b1189442c09646e554140b\',\n    \'validation\': \'0a1d85bc1de1fe1fec345600a02d3329af2e3e8827c1536a6d08002463a7958d\',\n}\nBUSINESS = re.compile(r\'\\b(business|entrepreneur\\w*|customer\\w*|market\\w*|profit\\w*|revenue|finance|financial|banking|loan\\w*|income|expense\\w*|investment\\w*|economy|economic\\w*|sales|retail|commerce|company|companies|industry|manufactur\\w*|costs?|prices?)\\b\', re.I)\n\ndef normalized(text):\n    return \' \'.join(text.casefold().split())\n\ndef download(split, cache):\n    path = cache / f\'qnli-{REVISION}-{split}.jsonl\'\n    if not path.exists() or hashlib.sha256(path.read_bytes()).hexdigest() != HASHES[split]:\n        request = urllib.request.Request(f\'{BASE}/{split}.jsonl\', headers={\'User-Agent\':\'STRIVE-research-dataset-preparation/1.0\'})\n        temporary = path.with_suffix(\'.download\')\n        try:\n            with urllib.request.urlopen(request, timeout=90) as response, temporary.open(\'wb\') as output:\n                while chunk := response.read(1024 * 1024):\n                    output.write(chunk)\n            if hashlib.sha256(temporary.read_bytes()).hexdigest() != HASHES[split]:\n                raise ValueError(f\'Unexpected {split} checksum. Dataset download was not accepted.\')\n            temporary.replace(path)\n        finally:\n            temporary.unlink(missing_ok=True)\n    return [json.loads(line) for line in path.read_text(encoding=\'utf-8\').splitlines() if line.strip()]\n\ndef convert(rows, split):\n    selected = []\n    seen = set()\n    for row in rows:\n        if row.get(\'label\') not in (0, 1) or type(row.get(\'label\')) is not int:\n            continue\n        question, answer = str(row.get(\'text1\', \'\')).strip(), str(row.get(\'text2\', \'\')).strip()\n        if not BUSINESS.search(question) or not answer or len(answer) > 3500:\n            continue\n        source_label = \'entailment\' if row[\'label\'] == 0 else \'not entailment\'\n        if row.get(\'label_text\') != source_label:\n            raise ValueError(\'The pinned dataset label mapping changed. Stop and review it.\')\n        signature = (normalized(question), normalized(answer))\n        if signature in seen:\n            continue\n        seen.add(signature)\n        selected.append({\'businessContext\':\'Business/economics question-answer relevance; no applicant facts.\',\n                         \'question\':question, \'answer\':answer, \'label\':row[\'label\'], \'language\':\'en-IN\',\n                         \'sourceDataset\':DATASET, \'sourceRevision\':REVISION, \'sourceSplit\':split,\n                         \'sourceIndex\':row[\'idx\'], \'sourceLabel\':row[\'label\'], \'sourceLabelText\':source_label,\n                         \'weakAuxiliary\':True, \'sourceLicense\':\'Upstream SQuAD CC BY-SA 4.0\',\n                         \'domainTerms\':sorted(set(match.group(0).lower() for match in BUSINESS.finditer(question)))})\n    # Questions OR answers shared by records belong to one component, even after sampling.\n    parents = list(range(len(selected)))\n    def root(i):\n        while parents[i] != i:\n            parents[i] = parents[parents[i]]\n            i = parents[i]\n        return i\n    keys = {}\n    for i, row in enumerate(selected):\n        for prefix, key in [(\'q\', row[\'question\']), (\'a\', row[\'answer\'])]:\n            identity = (prefix, normalized(key))\n            if identity in keys:\n                parents[root(i)] = root(keys[identity])\n            keys[identity] = i\n    components = collections.defaultdict(list)\n    for i, row in enumerate(selected):\n        components[root(i)].append(hashlib.sha256((normalized(row[\'question\'])+\'\\n\'+normalized(row[\'answer\'])).encode()).hexdigest())\n    for i, row in enumerate(selected):\n        row[\'groupId\'] = f\'qnli-{split}-\' + min(components[root(i)])[:16]\n    return selected\n\ndef balanced_sample(rows, maximum):\n    selected = []\n    per_class = min(maximum // 2, *(sum(row[\'label\'] == label for row in rows) for label in (0, 1)))\n    if per_class < 5:\n        raise ValueError(\'Too few real labelled auxiliary examples; no fabricated pairs were substituted.\')\n    for label in (0, 1):\n        candidates = [row for row in rows if row[\'label\'] == label]\n        candidates.sort(key=lambda row: hashlib.sha256((row[\'question\']+\'\\n\'+row[\'answer\']).encode()).hexdigest())\n        selected.extend(candidates[:per_class])\n    return selected\n\ndef remove_overlap(rows, reference):\n    questions = {normalized(row[\'question\']) for row in reference}\n    answers = {normalized(row[\'answer\']) for row in reference}\n    return [row for row in rows if normalized(row[\'question\']) not in questions and normalized(row[\'answer\']) not in answers]\n\ndef prepare(output=\'data/training\', maximum=600, validation_maximum=120):\n    if not 20 <= maximum <= 3000 or not 10 <= validation_maximum <= 1000:\n        raise ValueError(\'Use 20–3000 auxiliary training rows and 10–1000 validation rows.\')\n    output = Path(output)\n    cache = output / \'external-cache\'\n    cache.mkdir(parents=True, exist_ok=True)\n    training_candidates = convert(download(\'train\', cache), \'train\')\n    validation_candidates = convert(download(\'validation\', cache), \'validation\')\n    training = balanced_sample(training_candidates, maximum)\n    validation = balanced_sample(remove_overlap(validation_candidates, training_candidates), validation_maximum)\n    manifest = {\'dataset\':DATASET, \'revision\':REVISION, \'upstreamLicense\':\'CC BY-SA 4.0 (SQuAD)\',\n                \'datasetCard\':f\'https://huggingface.co/datasets/{DATASET}/blob/{REVISION}/README.md\',\n                \'upstreamLicenseSource\':\'https://rajpurkar.github.io/SQuAD-explorer/\',\n                \'taskSource\':\'https://huggingface.co/datasets/nyu-mll/glue#qnli\',\n                \'labelMeaning\':{\'0\':\'contains an answer\',\'1\':\'does not contain an answer; not necessarily unrelated\'},\n                \'auxiliaryOnly\':True, \'notEntrepreneurOutcomeData\':True,\n                \'selection\':\'Question-only business/economics keyword filter, deterministic balanced sample; no generated negative pairs.\',\n                \'modifications\':\'Filtered, sampled, converted field names and added provenance and connected-component groups for STRIVE; 2026-10-18.\',\n                \'files\':{}}\n    for split, rows, candidates in [(\'train\',training,training_candidates),(\'validation\',validation,validation_candidates)]:\n        path = output / f\'external-qnli-{split}.jsonl\'\n        content = \'\'.join(json.dumps(row,ensure_ascii=False)+\'\\n\' for row in rows)\n        path.write_text(content,encoding=\'utf-8\')\n        manifest[\'files\'][split] = {\'path\':str(path), \'rows\':len(rows), \'filteredCandidates\':len(candidates),\n                                   \'groups\':len({row[\'groupId\'] for row in rows}), \'classCounts\':dict(collections.Counter(row[\'label\'] for row in rows)),\n                                   \'sourceSha256\':HASHES[split], \'preparedSha256\':hashlib.sha256(content.encode()).hexdigest()}\n    (output/\'external-data-manifest.json\').write_text(json.dumps(manifest,indent=2)+\'\\n\',encoding=\'utf-8\')\n    print(json.dumps(manifest,indent=2))\n    return training, validation, manifest\n\nif __name__ == \'__main__\':\n    parser=argparse.ArgumentParser(description=__doc__)\n    parser.add_argument(\'--output\',default=\'data/training\')\n    parser.add_argument(\'--max-train\',type=int,default=600)\n    parser.add_argument(\'--max-validation\',type=int,default=120)\n    args=parser.parse_args()\n    prepare(args.output,args.max_train,args.max_validation)\n', 'cloud_train_external.py': '"""Run only in a hosted Google Colab runtime; no local model training."""\nimport importlib.util\nif importlib.util.find_spec(\'google.colab\') is None:\n    raise RuntimeError(\'Run external_cloud_training.ipynb in hosted Google Colab. Local training is disabled.\')\n\nimport collections\nimport datetime\nimport hashlib\nimport json\nimport time\nfrom pathlib import Path\nimport numpy as np\nfrom sklearn.linear_model import LogisticRegression\nfrom sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, confusion_matrix\nfrom sklearn.model_selection import StratifiedGroupKFold\nfrom train_cloud import check_training_file, features, embedding, EMBEDDING_MODEL, DIMENSION, FEATURE_VERSION\nfrom external_data import prepare, remove_overlap\n\ndef evaluate_labels(labels, probabilities):\n    labels, probabilities = np.asarray(labels), np.asarray(probabilities)\n    valid = np.isfinite(probabilities)\n    truth, scores = labels[valid], probabilities[valid]\n    if not len(truth):\n        raise ValueError(\'No valid held-out predictions; model promotion is disabled.\')\n    predicted, flags = (scores >= .5).astype(int), scores >= .75\n    return {\'rows\':len(labels), \'evaluatedRows\':int(valid.sum()), \'evaluationCoverage\':float(valid.mean()),\n            \'accuracy\':float(accuracy_score(truth,predicted)), \'macroF1\':float(f1_score(truth,predicted,labels=[0,1],average=\'macro\',zero_division=0)),\n            \'offTopicRecall\':float(recall_score(truth,predicted,zero_division=0)),\n            \'offTopicPrecision\':float(precision_score(truth,predicted,zero_division=0)),\n            \'confusionMatrix\':confusion_matrix(truth,predicted,labels=[0,1]).tolist(),\n            \'flaggedRows\':int(flags.sum()), \'flagCoverage\':float(flags.mean()),\n            \'flagPrecision\':float(truth[flags].mean()) if flags.any() else None,\n            \'abstentionRate\':float(((scores>.25)&(scores<.75)).mean())}\n\ndef promotion_decision(baseline, candidate):\n    checks = {\n        \'completeGroupedEvaluation\':baseline[\'evaluationCoverage\']==1 and candidate[\'evaluationCoverage\']==1,\n        \'macroF1ImprovedByAtLeastTwoPoints\':candidate[\'macroF1\'] >= baseline[\'macroF1\']+.02,\n        \'offTopicRecallNotWorse\':candidate[\'offTopicRecall\'] >= baseline[\'offTopicRecall\'],\n        \'offTopicPrecisionNotWorse\':candidate[\'offTopicPrecision\'] >= baseline[\'offTopicPrecision\'],\n        \'usefulHighThresholdFlags\':candidate[\'flaggedRows\'] >= 3 and (candidate[\'flagPrecision\'] or 0) >= .8,\n    }\n    return {\'recommendCandidate\':all(checks.values()),\'checks\':checks,\n            \'scope\':\'Experimental grouped-CV recommendation. A fresh expert-reviewed target test set is still needed.\'}\n\ndef fit_model(target_x, target_y, external_x=None, external_y=None):\n    model=LogisticRegression(C=1.0,class_weight=\'balanced\',max_iter=3000,random_state=42)\n    if external_x is None:\n        return model.fit(target_x,target_y)\n    # Bound the aggregate influence of auxiliary labels, regardless of their row count.\n    weights=np.concatenate([np.ones(len(target_y)),np.full(len(external_y),.5*len(target_y)/len(external_y))])\n    return model.fit(np.concatenate([target_x,external_x]),np.concatenate([target_y,external_y]),sample_weight=weights)\n\ndef cached_vectors(rows, api_key, cache):\n    saved=json.loads(cache.read_text(encoding=\'utf-8\')) if cache.exists() else {}\n    vectors=[]\n    for index,row in enumerate(rows):\n        cache_key=hashlib.sha256((EMBEDDING_MODEL+FEATURE_VERSION+features(row)).encode()).hexdigest()\n        if cache_key not in saved:\n            saved[cache_key]=embedding(features(row),api_key).tolist()\n            temporary=cache.with_suffix(\'.tmp\')\n            temporary.write_text(json.dumps(saved),encoding=\'utf-8\')\n            temporary.replace(cache)\n            time.sleep(.75)\n        vector=np.asarray(saved[cache_key],dtype=float)\n        if vector.shape != (DIMENSION,) or not np.isfinite(vector).all() or not np.isclose(np.linalg.norm(vector),1,atol=1e-5):\n            raise ValueError(\'Cached embedding is invalid. Remove that cache entry; no fake vector was substituted.\')\n        vectors.append(vector)\n        if (index+1)%50==0: print(f\'Hosted embeddings ready: {index+1}/{len(rows)}\')\n    return np.asarray(vectors)\n\ndef artifact(model, rows, metrics, suffix):\n    data_hash=hashlib.sha256(json.dumps(rows,sort_keys=True,ensure_ascii=False).encode()).hexdigest()\n    return {\'format\':\'strive-relevance-v1\',\'modelVersion\':f\'relevance-{data_hash[:12]}-{suffix}\',\n            \'featureVersion\':FEATURE_VERSION,\'embeddingModel\':EMBEDDING_MODEL,\'dimension\':DIMENSION,\n            \'coefficients\':model.coef_[0].tolist(),\'intercept\':float(model.intercept_[0]),\'classes\':[0,1],\n            \'trainingDataHash\':data_hash,\'trainedAt\':datetime.datetime.now(datetime.timezone.utc).isoformat(),\n            \'metrics\':metrics,\'advisoryOnly\':True}\n\ndef train_with_external(path, api_key, maximum_external=600):\n    target=check_training_file(path)\n    if any(row.get(\'weakAuxiliary\') or row.get(\'sourceDataset\')==\'SetFit/qnli\' for row in target):\n        raise ValueError(\'Upload the reviewed target answers separately. External rows must never be counted as reviewed business evidence.\')\n    external, validation, manifest=prepare(\'external-data\',maximum_external,120)\n    external=remove_overlap(external,target)\n    validation=remove_overlap(validation,target+external)\n    if min(collections.Counter(row[\'label\'] for row in external).values(),default=0)<5:\n        raise ValueError(\'Too few external examples remain after removing overlap.\')\n    all_rows=target+external+validation\n    print(f\'Using {len(target)} reviewed target answers, {len(external)} auxiliary training pairs, and {len(validation)} separate auxiliary validation pairs.\')\n    print(\'QNLI negatives mean does-not-answer, not necessarily off-topic. Target metrics stay separate.\')\n    matrix=cached_vectors(all_rows,api_key,Path(\'embedding_cache.json\'))\n    target_x=matrix[:len(target)]; external_x=matrix[len(target):len(target)+len(external)]; validation_x=matrix[len(target)+len(external):]\n    target_y=np.array([row[\'label\'] for row in target]); external_y=np.array([row[\'label\'] for row in external])\n    groups=np.array([row[\'groupId\'] for row in target])\n    predictions={name:np.full(len(target),np.nan) for name in [\'baseline\',\'externalCandidate\']}\n    folds=[]\n    splitter=StratifiedGroupKFold(n_splits=min(5,len(set(groups))),shuffle=True,random_state=42)\n    for fold,(train_ids,test_ids) in enumerate(splitter.split(target_x,target_y,groups)):\n        if len(set(target_y[train_ids]))<2:\n            folds.append({\'fold\':fold,\'skipped\':\'Target training partition has only one class.\'});continue\n        assert not set(groups[train_ids]) & set(groups[test_ids])\n        baseline=fit_model(target_x[train_ids],target_y[train_ids])\n        candidate=fit_model(target_x[train_ids],target_y[train_ids],external_x,external_y)\n        predictions[\'baseline\'][test_ids]=baseline.predict_proba(target_x[test_ids])[:,1]\n        predictions[\'externalCandidate\'][test_ids]=candidate.predict_proba(target_x[test_ids])[:,1]\n        folds.append({\'fold\':fold,\'trainRows\':len(train_ids),\'testRows\':len(test_ids),\'testGroups\':sorted(set(groups[test_ids]))})\n    baseline_metrics=evaluate_labels(target_y,predictions[\'baseline\'])\n    candidate_metrics=evaluate_labels(target_y,predictions[\'externalCandidate\'])\n    decision=promotion_decision(baseline_metrics,candidate_metrics)\n    comparison={\'evaluation\':\'Same held-out business-family folds for target-only baseline and auxiliary-data candidate; development selection, not an independent final test.\',\n                \'baseline\':baseline_metrics,\'externalCandidate\':candidate_metrics,\'promotion\':decision,\'folds\':folds,\n                \'targetRows\':len(target),\'targetGroups\':len(set(groups)), \'auxiliaryTrainingRows\':len(external),\n                \'auxiliaryValidationRows\':len(validation),\'externalData\':manifest,\n                \'limitations\':[\'Only five target families in the supplied prepared file; uncertainty is high.\',\'Assistant-reviewed target labels are not independent expert ground truth.\',\'English auxiliary data does not establish regional-language accuracy.\',\'No model predicts business success or certifies factual truth.\']}\n    baseline_final=fit_model(target_x,target_y)\n    candidate_final=fit_model(target_x,target_y,external_x,external_y)\n    comparison[\'auxiliaryValidation\']={name:evaluate_labels([r[\'label\'] for r in validation],model.predict_proba(validation_x)[:,1]) for name,model in [(\'baseline\',baseline_final),(\'externalCandidate\',candidate_final)]}\n    for name,metrics in [(\'baseline\',baseline_metrics),(\'externalCandidate\',candidate_metrics)]:\n        metrics.update({\'evaluation\':comparison[\'evaluation\'],\'groups\':len(set(groups)),\'advisoryThreshold\':.75,\n                        \'classCounts\':dict(collections.Counter(row[\'label\'] for row in target)),\n                        \'externalRows\':0 if name==\'baseline\' else len(external),\'selection\':decision,\'languages\':{}})\n    baseline_artifact=artifact(baseline_final,target,baseline_metrics,\'baseline\')\n    candidate_artifact=artifact(candidate_final,target+external,candidate_metrics,\'external\')\n    selected=candidate_artifact if decision[\'recommendCandidate\'] else baseline_artifact\n    selected_model=candidate_final if decision[\'recommendCandidate\'] else baseline_final\n    for name,value in [(\'baseline-model.json\',baseline_artifact),(\'candidate-model.json\',candidate_artifact),(\'model.json\',selected),(\'evaluation.json\',selected[\'metrics\']),(\'comparison.json\',comparison)]:\n        Path(name).write_text(json.dumps(value,indent=2),encoding=\'utf-8\')\n    parity={\'vectors\':target_x[:5].tolist(),\'expectedOffTopicScores\':selected_model.predict_proba(target_x[:5])[:,1].tolist()}\n    Path(\'parity-vectors.json\').write_text(json.dumps(parity,indent=2),encoding=\'utf-8\')\n    print(json.dumps({\'targetBaseline\':baseline_metrics,\'targetCandidate\':candidate_metrics,\'recommendation\':decision},indent=2))\n    print(\'Saved model.json for the selected model. candidate-model.json remains experimental unless all promotion checks pass. No deployment was performed.\')\n    return comparison\n'}
for name, source in SOURCES.items():
    Path(name).write_text(source, encoding="utf-8")


In [ ]:
from google.colab import files
from train_cloud import check_training_file
uploaded = files.upload()
paths = [name for name in uploaded if name.endswith('.jsonl') and 'external' not in name]
if len(paths) != 1:
    raise ValueError('Upload exactly one reviewed target JSONL, plus an optional embedding_cache.json. Do not upload external data as target labels.')
rows = check_training_file(paths[0])
print(f'Target preflight passed: {len(rows)} answers. External data downloads automatically next.')


In [ ]:
from google.colab import userdata
from getpass import getpass
try:
    api_key = userdata.get('GEMINI_API_KEY')
except Exception:
    api_key = getpass('Gemini API key for hosted embeddings (hidden): ')
if not api_key:
    raise ValueError('A Gemini embedding API key is required. Groq generation keys cannot replace embedding features.')


In [ ]:
from cloud_train_external import train_with_external
try:
    comparison = train_with_external(paths[0], api_key, maximum_external=600)
finally:
    api_key = None


In [ ]:
import zipfile
outputs = ['model.json','evaluation.json','parity-vectors.json','comparison.json','baseline-model.json','candidate-model.json','external-data/external-data-manifest.json']
with zipfile.ZipFile('strive-cloud-training-results.zip','w',zipfile.ZIP_DEFLATED) as archive:
    for name in outputs:
        archive.write(name,Path(name).name)
files.download('strive-cloud-training-results.zip')
print('Keep comparison.json with the artifact. The current deployed model is unchanged until you choose to deploy the selected artifact.')


## If interrupted
Re-run using the same hosted runtime to reuse `embedding_cache.json`. Before disconnecting, optionally download that cache to reuse it in your next private run. API quota failures stop without fabricating vectors. Upload the cache alongside your reviewed target JSONL on the next run.

`model.json` is the selected result: the baseline stays selected unless the candidate improves target macro-F1 by at least two points, does not reduce off-topic precision/recall, has complete grouped evaluation, and produces at least three high-threshold flags with at least 80% observed precision. This small grouped evaluation is an experimental selection, not a fresh independent test. Do not claim an accuracy increase until the measured comparison supports it. No hosting/deployment is started by this notebook.